# MIC smoke test on ROSE (50 iterations)

Before running: Settings > Accelerator: **GPU T4 x1**, Internet: **on**.
Attach two datasets: the ROSE MIC data (contents of `data/mic/`) and `mit_b5.pth`.
Add-ons > Secrets: `GITHUB_TOKEN` (only needed if the repo is private).
Set `REPO` below to your GitHub repo.

In [ ]:
REPO = "Muhammad-Zubair-Yousafzai/weedrefine-uda"
PAIR_CFG = "configs/rose/bean_2019_to_2021_s0_smoke.py"
import os
# Kaggle's default matplotlib backend is missing in the MIC env; MIC only saves figures.
os.environ["MPLBACKEND"] = "Agg"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None   # public repo
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

def git(*args):
    # never print the token, even in errors
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError((r.stderr or r.stdout).replace(token or "\0", "***"))

if not os.path.isdir("weedrefine-uda"):
    git("clone", "-q", url, "weedrefine-uda")
    git("-C", "weedrefine-uda", "remote", "set-url", "origin", f"https://github.com/{REPO}.git")
else:
    git("-C", "weedrefine-uda", "pull", "-q", url, "main")   # get fixes pushed since the clone
os.chdir("/kaggle/working/weedrefine-uda")
!git log -1 --format='%h %s'

In [ ]:
# About 5 to 10 minutes. Ends with 'Setup done' and a config check line.
!bash scripts/kaggle_mic_setup.sh

In [ ]:
# 50 training iterations, then evaluation on the 25 target val images.
!cd third_party/MIC/seg && /root/miniforge/envs/mic/bin/python run_experiments.py --config {PAIR_CFG} 2>&1 | tail -n 80

In [ ]:
# Where the log went, peak GPU memory and the evaluation lines.
!ls -t third_party/MIC/seg/work_dirs/local-rose/ | head -3
!LOG=$(ls -t third_party/MIC/seg/work_dirs/local-rose/*/*.log | head -1); echo $LOG; grep -E 'memory|mIoU|weed|crop|background' $LOG | tail -n 25